In [3]:
import lightgbm as lgb
import pandas as pd
from sklearn.metrics import average_precision_score

from fraudguard.config import load_settings
from fraudguard.data.load import load_dataset
from fraudguard.data.split import temporal_split

settings = load_settings()
df = load_dataset(settings)
split = temporal_split(df, settings.split)

EXCLUDE = ["TransactionID", "TransactionDT", "isFraud"]
features = [c for c in df.columns if c not in EXCLUDE]
cat_cols = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]

# Categories learned from train only (valid categories unseen in train become NaN)
categories = {c: split.train[c].dropna().unique() for c in cat_cols}


def prepare(part: pd.DataFrame) -> pd.DataFrame:
    X = part[features].copy()
    for c in cat_cols:
        X[c] = pd.Categorical(X[c], categories=categories[c])
    return X


X_train, y_train = prepare(split.train), split.train["isFraud"]
X_valid, y_valid = prepare(split.valid), split.valid["isFraud"]

PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    random_state=settings.seed,
    verbose=-1,
)


def fit_eval(cols: list[str]) -> tuple[lgb.LGBMClassifier, float]:
    model = lgb.LGBMClassifier(**PARAMS)
    model.fit(
        X_train[cols],
        y_train,
        eval_set=[(X_valid[cols], y_valid)],
        eval_metric="average_precision",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    score = average_precision_score(y_valid, model.predict_proba(X_valid[cols])[:, 1])
    return model, score


full_model, full_score = fit_eval(features)
print(f"{len(features)} features ({len(cat_cols)} categorical) -> valid PR-AUC: {full_score:.4f}")

C:\Users\fblas\AppData\Local\Temp\ipykernel_12672\885699921.py:22: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = pd.Categorical(X[c], categories=categories[c])
C:\Users\fblas\AppData\Local\Temp\ipykernel_12672\885699921.py:22: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = pd.Categorical(X[c], categories=categories[c])
C:\Users\fblas\AppData\Local\Temp\ipykernel_12672\885699921.py:22: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = pd.Categorical(X[c], categories=categories[c])
C:\Users\fblas\AppData\Local\Temp\ipykernel_12672\885699921.py:22: Pandas4Warning: Constructing a Categ

431 features (31 categorical) -> valid PR-AUC: 0.6069


In [4]:
importance = pd.Series(full_model.booster_.feature_importance("gain"), index=features).sort_values(
    ascending=False
)
share = importance / importance.sum()

print(f"Top 40 features explain {share.head(40).sum():.1%} of total gain")
share.head(40).round(4)

Top 40 features explain 68.7% of total gain


V258              0.1015
C1                0.0480
C13               0.0429
DeviceInfo        0.0408
card1             0.0287
card2             0.0281
TransactionAmt    0.0255
addr1             0.0245
C14               0.0234
R_emaildomain     0.0196
D2                0.0184
id_31             0.0158
P_emaildomain     0.0154
V294              0.0152
D1                0.0129
D15               0.0128
D4                0.0119
V257              0.0117
C2                0.0113
V242              0.0111
card5             0.0109
C11               0.0106
C8                0.0104
card6             0.0102
D10               0.0102
V317              0.0097
dist1             0.0092
C6                0.0091
V149              0.0091
C7                0.0089
card3             0.0081
V70               0.0078
V295              0.0073
id_33             0.0071
id_30             0.0070
D8                0.0066
C4                0.0064
D3                0.0064
id_02             0.0062
C10               0.0061


In [5]:
results = {}
for n in (20, 40, 60, 100):
    _, score = fit_eval(list(importance.index[:n]))
    results[n] = score
    print(f"top {n:>3}: PR-AUC {score:.4f} ({score / full_score - 1:+.1%} vs all)")

c:\dev\fraudguard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


top  20: PR-AUC 0.5318 (-12.4% vs all)


c:\dev\fraudguard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


top  40: PR-AUC 0.5884 (-3.0% vs all)


c:\dev\fraudguard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


top  60: PR-AUC 0.6114 (+0.7% vs all)


c:\dev\fraudguard\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


top 100: PR-AUC 0.5982 (-1.4% vs all)


In [6]:
import yaml

from fraudguard.config import PROJECT_ROOT

N_FEATURES = 60
selected = list(importance.index[:N_FEATURES])
feature_config = {
    "numeric": [c for c in selected if c not in cat_cols],
    "categorical": [c for c in selected if c in cat_cols],
}

with open(PROJECT_ROOT / "configs" / "features.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(feature_config, f, sort_keys=False)

print(f"{len(feature_config['numeric'])} numeric, {len(feature_config['categorical'])} categorical")
print("Categorical:", feature_config["categorical"])

50 numeric, 10 categorical
Categorical: ['DeviceInfo', 'R_emaildomain', 'id_31', 'P_emaildomain', 'card6', 'id_33', 'id_30', 'M5', 'M4', 'M6']


## Conclusions

**Method.** A LightGBM model was trained on all 431 candidate features (excluding `TransactionID` and `TransactionDT`) on the train set, and features were ranked by gain importance. The model was then retrained using only the top-N features and evaluated on the validation set (PR-AUC).

| Features | Valid PR-AUC | vs all |
| --- | --- | --- |
| 20 | 0.532 | −12.4% |
| 40 | 0.588 | −3.0% |
| 60 | 0.611 | +0.7% |
| 100 | 0.598 | −1.4% |
| 431 (all) | 0.607 | — |

**Decision: keep the top 60 features** (50 numeric, 10 categorical), saved in `configs/features.yaml`. It is the smallest set whose performance is indistinguishable from using all features: the differences between 60, 100 and 431 (±1–2%) are within run-to-run noise (row subsampling, early stopping), so they should not be read as real improvements. Using 40 features loses ~3%, beyond that noise. This removes 86% of the columns, which simplifies the API schema and drift monitoring.

**Notes.**
- The ranking is dominated by card-related counters (`C*`), card and address fields, transaction amount, time-delta features (`D*`) and email domains. `ProductCD` is not in the top 60 despite its strong association with fraud: its information is largely carried by other features (e.g. identity missingness patterns).
- `DeviceInfo` ranks 4th but is a high-cardinality categorical; gain importance tends to be inflated for such features and it may overfit. Its contribution should be checked on the future set.
- Early stopping used the validation set, so validation PR-AUC is slightly optimistic. It is used here only to compare feature subsets; the final reported metric will be on the future set.